In [1]:
import json
import re
import xml.etree.ElementTree as ET

from pathlib import Path
from typing import Tuple

In [2]:
def get_filename(path: str | Path) -> str:
    return str(path).split("/")[-1]


def load_file(file_path: str | Path) -> str:
    with open(file_path, "r", encoding="latin1") as f:
        return f.read().encode("utf-8")


def clean_span(dirty_span: str) -> str:
    cleaner = "<.*?>"
    return re.sub(cleaner, "", str(dirty_span))


def parse_xml(raw_xml: ET.Element) -> str:
    text = ""
    for section in raw_xml:
        # Title is the span with ID '0'
        if "sid" in section.attrib and section.attrib["sid"] == "0":
            text += f"Title: {section.text}\n"
            continue

        # Extract the information from the rest of the paper
        section_tag = section.tag
        if section_tag == "ABSTRACT":
            text += "\nABSTRACT\n"
        else:
            attribs = section.attrib
            section_number = f" {attribs['number']}" if "number" in attribs else ""
            section_title = attribs["title"] if "title" in attribs else ""
            text += f"\n{section_tag}{section_number}: {section_title}\n"

        for span in section:
            text += f"{span.text}\n"
    return text


def load_xml(paper_idx: str, paper_path: Path) -> str:
    xml_path = paper_path.joinpath("Reference_XML", f"{paper_idx}.xml")
    raw_xml = ET.fromstring(load_file(xml_path))
    parsed_xml = parse_xml(raw_xml)
    return parsed_xml

In [3]:
def clean_abstract(raw_abstract: str) -> str:
    raw_spans = raw_abstract.splitlines()
    spans = [clean_span(rs.decode("utf-8")) for rs in raw_spans]
    return " ".join(spans)


def load_summaries(paper_path: Path) -> Tuple[str, str]:
    summ_path = paper_path.joinpath("summary")
    summ_files = [f for f in summ_path.iterdir()]
    for sf in summ_files:
        if "abstract" in get_filename(sf):
            raw_abstract = load_file(sf)
            abstract = clean_abstract(raw_abstract)
        if "human" in get_filename(sf):
            summary = load_file(sf).decode("utf-8")
    return abstract, summary

In [4]:
def get_papers_map(curr_dir: Path) -> dict[str:Path]:
    return {
        get_filename(paper_path): paper_path
        for paper_path in curr_dir.iterdir()
        if paper_path.is_dir()
    }

In [5]:
def extract_title(text):
    """
    Extract the title from the text field.
    The title starts with "Title:" and ends before "ABSTRACT" or "SECTION 1:"
    """
    # Find the title using regex
    # Match "Title:" followed by any text until we hit "ABSTRACT" or "SECTION 1:"
    match = re.search(r'Title:\s*(.*?)\s*(?:ABSTRACT|SECTION 1:)', text, re.DOTALL)
    
    if match:
        title = match.group(1).strip()
        # Remove any extra whitespace and newlines
        title = ' '.join(title.split())
        return title
    return None

In [6]:
base_dir = Path("raw_data")
train_2019_path = base_dir.joinpath("Training-Set-2019")
others_path = [d for d in base_dir.iterdir() if d.is_dir() and d != train_2019_path]

In [7]:
# First parse all the data from 2019, Task 2
all_papers = {}
dirs_to_check = [d for d in train_2019_path.iterdir() if d.is_dir()]

for curr_dir in dirs_to_check:
    # Create map paper_id: Path
    papers_map = get_papers_map(curr_dir)

    # Extract data from each paper
    for paper_idx in papers_map:
        # Add new entry
        if paper_idx not in all_papers:
            all_papers[paper_idx] = {"full_text": "", "abstract": "", "human_summary": ""}
            # Save full text
            all_papers[paper_idx]["full_text"] = load_xml(paper_idx, papers_map[paper_idx])
            # Get abstract and human summary
            abstract, summary = load_summaries(papers_map[paper_idx])
            all_papers[paper_idx]["abstract"] = abstract
            all_papers[paper_idx]["human_summary"] = summary
            # Extract title
            all_papers[paper_idx]["title"] = extract_title(all_papers[paper_idx]["full_text"])

In [8]:
print(len(all_papers))

1018


In [9]:
# Then go and check in the past years if there are "new" documents
for other in others_path:
    papers_map = get_papers_map(other)

    # Extract data from each paper
    for paper_idx in papers_map:
        if paper_idx not in all_papers:
            all_papers[paper_idx] = {"full_text": "", "abstract": "", "human_summary": ""}
            # Save full text
            all_papers[paper_idx]["full_text"] = load_xml(paper_idx, papers_map[paper_idx])
            # Get abstract and human summary
            abstract, summary = load_summaries(papers_map[paper_idx])
            all_papers[paper_idx]["abstract"] = abstract
            all_papers[paper_idx]["human_summary"] = summary
            # Extract title
            all_papers[paper_idx]["title"] = extract_title(all_papers[paper_idx]["full_text"])

In [10]:
print(len(all_papers))

1038


In [14]:
count_no_title = 0
for paper_idx in all_papers:
    if all_papers[paper_idx]["title"] is None:
        print(all_papers[paper_idx])
        count_no_title += 1
    # print(all_papers[paper_idx].keys())
print(f">> Found {count_no_title} papers without a title.")

{'full_text': '\nABSTRACT\nFor developing a data-driven text rewriting algorithm for paraphrasing, it is essential to have a monolingual corpus of aligned paraphrased sentences.\nNews article headlines are a rich source of paraphrases; they tend to describe the same event in various different ways, and can easily be obtained from the web.\nWe compare two methods of aligning headlines to construct such an aligned corpus of paraphrases, one based on clustering, and the other on pairwise similarity-based matching.\nWe show that the latter performs best on the task of aligning paraphrastic headlines.\n\nSECTION 1: Introduction\nIn recent years, text-to-text generation has received increasing attention in the field of Natural Language Generation (NLG).\nIn contrast to traditional concept-to-text systems, text-to-text generation systems convert source text to target text, where typically the source and target text share the same meaning to some extent.\nApplications of text-to-text generatio

In [15]:
# Save to disk
def append_to_jsonl(data: dict, file_path: str | Path) -> None:
    with open(file_path, "a") as fp:
        fp.write(json.dumps(data))
        fp.write("\n")

out_path = base_dir.joinpath("cl_scisumm.jsonl")
for paper_idx in all_papers:
    to_save = {"paper_id": paper_idx}
    to_save.update(all_papers[paper_idx])
    append_to_jsonl(to_save, out_path)